In [33]:
from tinygrad.runtime.ops_clang import ClangProgram, ClangCompiler, MallocAllocator

In [34]:
out = MallocAllocator.alloc(4)
a = MallocAllocator.alloc(4)
b = MallocAllocator.alloc(4)

In [35]:
MallocAllocator.copyin(a, bytearray([2,0,0,0]))
MallocAllocator.copyin(b, bytearray([3,0,0,0]))

In [36]:
lib = ClangCompiler().compile("void add(int *out, int *a, int *b) {out[0] = a[0] + b[0]; }")

In [37]:
fxn = ClangProgram("add", lib)

In [38]:
fxn(out, a, b)


In [39]:
print(val:= MallocAllocator.as_buffer(out).cast("I").tolist()[0])

5


In [40]:
DEVICE = "CLANG"

In [41]:
import struct
from tinygrad.dtype import dtypes
from tinygrad.device import Buffer, Device
from tinygrad.ops import BinaryOps, MetaOps, UOp, UOps
from tinygrad.shape.shapetracker import ShapeTracker


In [42]:
out = Buffer(DEVICE, 1, dtypes.int32).allocate()
a = Buffer(DEVICE, 1, dtypes.int32).allocate().copyin(memoryview(bytearray(struct.pack("I", 2))))
b = Buffer(DEVICE, 1, dtypes.int32).allocate().copyin(memoryview(bytearray(struct.pack("I", 3))))


In [43]:
buf_1 = UOp(UOps.DEFINE_GLOBAL, dtypes.int32.ptr(), (), 1)
buf_2 = UOp(UOps.DEFINE_GLOBAL, dtypes.int32.ptr(), (), 2)
ld_1 = UOp(UOps.LOAD, dtypes.int32, (buf_1, ShapeTracker.from_shape((1,)).to_uop()))
ld_2 = UOp(UOps.LOAD, dtypes.int32, (buf_2, ShapeTracker.from_shape((1,)).to_uop()))
alu = ld_1 + ld_2
output_buf = UOp(UOps.DEFINE_GLOBAL, dtypes.int32.ptr(), (), 0)
st_0 = UOp(UOps.STORE, dtypes.void, (output_buf, ShapeTracker.from_shape((1,)).to_uop(),alu))
s = UOp(UOps.SINK, dtypes.void, (st_0,))

In [44]:
from tinygrad.engine.realize import get_kernel, CompiledRunner

In [45]:

kernel = get_kernel(Device[DEVICE].renderer, s).linearize()

In [46]:
fxn = CompiledRunner(kernel.to_program())
print(fxn.p.src)


void E_n2(int* restrict data0, int* restrict data1, int* restrict data2) {
  int val0 = *(data1+0);
  int val1 = *(data2+0);
  *(data0+0) = (val0+val1);
}


In [47]:
fxn.exec([out, a,b])

In [48]:
assert out.as_buffer().cast('I')[0] == 5

In [49]:
from tinygrad.engine.lazy import LazyBuffer
from tinygrad.engine.realize import run_schedule
from tinygrad.engine.schedule import create_schedule


In [50]:

# allocate some values + load in values
a = LazyBuffer.metaop(MetaOps.EMPTY, (1,), dtypes.int32, DEVICE)
b = LazyBuffer.metaop(MetaOps.EMPTY, (1,), dtypes.int32, DEVICE)
a.buffer.allocate().copyin(memoryview(bytearray(struct.pack("I", 2))))
b.buffer.allocate().copyin(memoryview(bytearray(struct.pack("I", 3))))
del a.srcs
del b.srcs


In [51]:
# describe the computation
out = a.alu(BinaryOps.ADD, b)

# schedule the computation as a list of kernels
sched = create_schedule([out])
for si in sched: print(si.ast.op)  # NOTE: the first two convert it to CLANG


UOps.SINK


In [52]:

# DEBUGGING: print the compute ast
print(sched[-1].ast)
# NOTE: sched[-1].ast is the same as st_0 above


UOp(UOps.SINK, dtypes.void, arg=None, src=(
  UOp(UOps.STORE, dtypes.void, arg=None, src=(
    UOp(UOps.DEFINE_GLOBAL, dtypes.int.ptr(), arg=0, src=()),
    x2:=UOp(UOps.VIEW, dtypes.void, arg=ShapeTracker(views=(View(shape=(1,), strides=(0,), offset=0, mask=None, contiguous=True),)), src=()),
    UOp(UOps.ALU, dtypes.int, arg=BinaryOps.ADD, src=(
      UOp(UOps.LOAD, dtypes.int, arg=None, src=(
        UOp(UOps.DEFINE_GLOBAL, dtypes.int.ptr(), arg=1, src=()),
         x2,)),
      UOp(UOps.LOAD, dtypes.int, arg=None, src=(
        UOp(UOps.DEFINE_GLOBAL, dtypes.int.ptr(), arg=2, src=()),
         x2,)),)),)),))


In [53]:

# run that schedule
run_schedule(sched)

# check the data out
assert out.realized.as_buffer().cast('I')[0] == 5


In [54]:
from tinygrad import Tensor

In [55]:
a = Tensor([2], dtype=dtypes.int32, device=DEVICE)
b = Tensor([3], dtype=dtypes.int32, device=DEVICE)
out = a + b


In [56]:
from tinygrad.engine.lazy import LazyBuffer
from tinygrad.engine.realize import run_schedule
from tinygrad.engine.schedule import create_schedule

In [57]:
from tinygrad.codegen.kernel import Kernel

In [58]:
a.lazydata.buffer.allocate

<bound method Buffer.allocate of <buf real:False device:CLANG size:1 dtype:dtypes.int offset:0>>

In [59]:
a

<Tensor <LB CLANG (1,) int (<MetaOps.COPY: 30>, None)> on CLANG with grad None>

In [60]:
d = Tensor([2], dtype=dtypes.int32, device=DEVICE)

In [61]:
e = Tensor([3], dtype=dtypes.int32, device=DEVICE)

In [62]:
out = d * e
out.realize

<bound method Tensor.realize of <Tensor <LB CLANG (1,) int (<BinaryOps.MUL: 10>, None)> on CLANG with grad None>>

In [63]:
out.device.lower

<function str.lower()>

In [64]:

from tinygrad.engine.lazy import LazyBuffer
from tinygrad.engine.realize import run_schedule
from tinygrad.engine.schedule import create_schedule

# allocate some values + load in values
a = LazyBuffer.metaop(MetaOps.EMPTY, (1,), dtypes.int32, DEVICE)
b = LazyBuffer.metaop(MetaOps.EMPTY, (1,), dtypes.int32, DEVICE)
a.buffer.allocate().copyin(memoryview(bytearray(struct.pack("I", 2))))
b.buffer.allocate().copyin(memoryview(bytearray(struct.pack("I", 3))))
del a.srcs
del b.srcs

# describe the computation
out = a.alu(BinaryOps.ADD, b)

# schedule the computation as a list of kernels
sched = create_schedule([out])
for si in sched: print(si.ast.op)  # NOTE: the first two convert it to CLANG

# DEBUGGING: print the compute ast
print(sched[-1].ast)
# NOTE: sched[-1].ast is the same as st_0 above

# run that schedule
run_schedule(sched)

# check the data out
assert out.realized.as_buffer().cast('I')[0] == 5

UOps.SINK
UOp(UOps.SINK, dtypes.void, arg=None, src=(
  UOp(UOps.STORE, dtypes.void, arg=None, src=(
    UOp(UOps.DEFINE_GLOBAL, dtypes.int.ptr(), arg=0, src=()),
    x2:=UOp(UOps.VIEW, dtypes.void, arg=ShapeTracker(views=(View(shape=(1,), strides=(0,), offset=0, mask=None, contiguous=True),)), src=()),
    UOp(UOps.ALU, dtypes.int, arg=BinaryOps.ADD, src=(
      UOp(UOps.LOAD, dtypes.int, arg=None, src=(
        UOp(UOps.DEFINE_GLOBAL, dtypes.int.ptr(), arg=1, src=()),
         x2,)),
      UOp(UOps.LOAD, dtypes.int, arg=None, src=(
        UOp(UOps.DEFINE_GLOBAL, dtypes.int.ptr(), arg=2, src=()),
         x2,)),)),)),))


In [65]:

out.alu

<bound method LazyBuffer.alu of <LB CLANG (1,) int (<BinaryOps.ADD: 9>, <buf real:True device:CLANG size:1 dtype:dtypes.int offset:0>)>>

In [67]:
c = a * b

In [72]:
run_schedule(c)

TypeError: object of type 'LazyBuffer' has no len()